In [24]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict
from datetime import datetime

In [25]:
model = YOLO("yolov10m.pt")  # COCO pretrained

In [26]:
VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}



def detect_vehicles(frame, conf_threshold=0.4):
    results = model(frame, conf=conf_threshold, verbose=False)
    
    vehicles = []
    
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            
            if cls_id in VEHICLE_CLASSES:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                vehicle_data = {
                    "bbox": [x1, y1, x2, y2],
                    "vehicle_type": VEHICLE_CLASSES[cls_id],
                    "confidence": round(conf, 3)
                }
                
                vehicles.append(vehicle_data)
    
    return vehicles

In [27]:
img_path = r"D:\Automatic ANPR\Self_Code\dataset_preprocessed\images\val\WB3.jpg"
frame = cv2.imread(img_path)

vehicles = detect_vehicles(frame)

for v in vehicles:
    x1, y1, x2, y2 = v["bbox"]
    label = f"{v['vehicle_type']} ({v['confidence']})"
    
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
    cv2.putText(frame, label, (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

cv2.imshow("Vehicle Detection", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("Structured Output:")
print(vehicles)

Structured Output:
[{'bbox': [0, 93, 121, 312], 'vehicle_type': 'car', 'confidence': 0.83}, {'bbox': [81, 33, 531, 611], 'vehicle_type': 'truck', 'confidence': 0.576}, {'bbox': [81, 33, 531, 611], 'vehicle_type': 'car', 'confidence': 0.413}]


## Video Inference

In [28]:
# Replace with your video path
video_path = r"C:\Users\100ra\Downloads\Traffic_sound_Indian_Traffic_Sounds_Traffic_Noise_Vehicle_Noise_Road_traffic_Shorts_Viral_cars_360P.mp4"

cap = cv2.VideoCapture(video_path)

camera_id = "CAM_01"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    vehicles = detect_vehicles(frame)

    # Draw detections
    for v in vehicles:
        x1, y1, x2, y2 = v["bbox"]
        label = f"{v['vehicle_type']} ({v['confidence']})"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        # Example structured log (ready for fusion later)
        structured_output = {
            "camera_id": camera_id,
            "vehicle_type": v["vehicle_type"],
            "confidence": v["confidence"],
            "timestamp": datetime.now().isoformat()
        }

        print(structured_output)

    cv2.imshow("Vehicle Type Detection - Video", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

{'camera_id': 'CAM_01', 'vehicle_type': 'car', 'confidence': 0.888, 'timestamp': '2026-02-14T11:59:07.966209'}
{'camera_id': 'CAM_01', 'vehicle_type': 'car', 'confidence': 0.835, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'car', 'confidence': 0.826, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'motorcycle', 'confidence': 0.805, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'truck', 'confidence': 0.714, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'car', 'confidence': 0.68, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'car', 'confidence': 0.662, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'motorcycle', 'confidence': 0.612, 'timestamp': '2026-02-14T11:59:07.966715'}
{'camera_id': 'CAM_01', 'vehicle_type': 'motorcycle', 'confidence': 0.581, 'timestamp': '2026-02-

## Hugging Face